# M1: Baseline Pencarian Lintas Modal

Kelompok 03 · IF25-40304 Pembelajaran Mesin Multimodal

> Notebook ini memuat kode yang sama dengan `src/m1_baseline.py`. Ubah kode di VSCode, lalu `git push`; di sini cukup dijalankan.

Encoder dibekukan (`clip-ViT-B-32` dan `clip-ViT-B-32-multilingual-v1`), tanpa pelatihan, dan berjalan di CPU.

*Data: SEA-VL Indonesian subset, CC BY-SA 4.0 (Cahyawijaya et al., 2025). Lihat `ATTRIBUTION.md`.*

## 1. Persiapan (khusus Google Colab)
Lewati sel ini jika dijalankan di komputer lokal.

In [ ]:
import os
if os.path.exists("/content/Tubes-Multimodal"):
    %cd /content/Tubes-Multimodal
    !git pull
else:
    %cd /content
    !git clone https://github.com/bayybrigas04/Tubes-Multimodal.git
    %cd /content/Tubes-Multimodal
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/Dataset-Tubes-Multimodal/bundle-mahasiswa.zip"
DATA_DIR = "/content/data/bundle-mahasiswa"
if not os.path.exists(DATA_DIR):
    !unzip -q "{ZIP_PATH}" -d /content/data
print(sorted(os.listdir(DATA_DIR)))

## 2. Konfigurasi
Di komputer lokal, `DATA_DIR` otomatis `data`. Di Colab, nilainya sudah diisi oleh sel persiapan.

In [ ]:
DATA_DIR = globals().get("DATA_DIR", "data")
OUT_DIR = "results/m1"
SEED = 42
KS = (1, 5, 10)
MAP_K = 10
ARAH = ["gambar->teks-ID", "gambar->teks-EN", "teks-ID->gambar", "teks-EN->gambar"]

## 3. Fungsi

In [ ]:
import os, re, json, random
import numpy as np, pandas as pd
from PIL import Image


def norm(s):
    return re.sub(r"[^\w\s]", "", str(s).lower()).strip()


def ranks_of_truth(A, B):
    """Peringkat (1 = terbaik) pasangan benar A[i]<->B[i] di antara semua B."""
    sim = A @ B.T
    diag = np.diag(sim)[:, None]
    return (sim > diag).sum(1) + 1


def metrik(A, B):
    """Recall@K dan mAP@K; satu item relevan per query, jadi AP@K = 1/rank jika rank<=K."""
    r = ranks_of_truth(A, B)
    out = {f"Recall@{k}": float((r <= k).mean()) for k in KS}
    out[f"mAP@{MAP_K}"] = float(np.where(r <= MAP_K, 1.0 / r, 0.0).mean())
    return out, r

In [ ]:
def muat_gambar(df, root):
    imgs = []
    for f in df["file"]:
        for p in (f, os.path.join(root, f), os.path.join(root, "images", os.path.basename(f))):
            if os.path.exists(p):
                imgs.append(Image.open(p).convert("RGB")); break
        else:
            raise FileNotFoundError(f)
    return imgs


def embed(df, root, cache):
    paths = {k: os.path.join(cache, f"{k}.npy") for k in ("img", "ti", "te")}
    if all(os.path.exists(p) for p in paths.values()):
        return {k: np.load(p) for k, p in paths.items()}
    from sentence_transformers import SentenceTransformer
    img_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
    txt_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
    E = img_model.encode(muat_gambar(df, root), batch_size=32, normalize_embeddings=True)
    Ti = txt_model.encode(list(df["caption_id"]), batch_size=64, normalize_embeddings=True)
    Te = img_model.encode(list(df["caption_en"]), batch_size=64, normalize_embeddings=True)
    os.makedirs(cache, exist_ok=True)
    for k, v in zip(("img", "ti", "te"), (E, Ti, Te)):
        np.save(paths[k], v)
    return {"img": E, "ti": Ti, "te": Te}

In [ ]:
def evaluasi(emb, mask=None):
    E, Ti, Te = emb["img"], emb["ti"], emb["te"]
    if mask is not None:
        E, Ti, Te = E[mask], Ti[mask], Te[mask]
    pasangan = {ARAH[0]: (E, Ti), ARAH[1]: (E, Te), ARAH[2]: (Ti, E), ARAH[3]: (Te, E)}
    hasil, ranks = {}, {}
    for nama, (A, B) in pasangan.items():
        hasil[nama], ranks[nama] = metrik(A, B)
    return pd.DataFrame(hasil).T, ranks


def contoh_query(df, emb, ranks_teks_id_ke_gambar, n=10, min_gagal=2, topk=3):
    sim = emb["ti"] @ emb["img"].T
    top = np.argsort(-sim, axis=1)[:, :topk]
    gagal = [i for i in range(len(df)) if top[i, 0] != i]
    rng = random.Random(SEED)
    pilih = rng.sample(gagal, min(min_gagal, len(gagal)))
    sisa = [i for i in range(len(df)) if i not in pilih]
    pilih += rng.sample(sisa, n - len(pilih))
    rng.shuffle(pilih)
    baris = []
    for i in pilih:
        baris.append({
            "query_id": df["caption_id"].iloc[i],
            "benar": os.path.basename(df["file"].iloc[i]),
            "top1": os.path.basename(df["file"].iloc[top[i, 0]]),
            "top2": os.path.basename(df["file"].iloc[top[i, 1]]),
            "top3": os.path.basename(df["file"].iloc[top[i, 2]]),
            "top1_benar": bool(top[i, 0] == i),
            "peringkat_benar": int(ranks_teks_id_ke_gambar[i]),
            "kategori": df["category"].iloc[i] if "category" in df else "",
        })
    return pd.DataFrame(baris)

## 4. Muat data dan hitung embedding

In [ ]:
np.random.seed(SEED); random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

val = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
N = len(val)
emb = embed(val, DATA_DIR, os.path.join(OUT_DIR, "cache_val"))
print(f"N val = {N}; pembanding acak Recall@1 = {1/N:.4f}")

## 5. Recall@K dan mAP@K (empat arah)

In [ ]:
tabel, ranks = evaluasi(emb)
tabel.to_csv(os.path.join(OUT_DIR, "tabel_recall_val.csv"), float_format="%.4f")
tabel.round(4)

## 6. Sensitivitas leakage
Tabel yang sama setelah membuang baris val yang captionnya juga ada di train.

In [ ]:
tr = set(norm(c) for c in train["caption_id"]) | set(norm(c) for c in train["caption_en"])
bocor = np.array([(norm(i) in tr) or (norm(e) in tr)
                  for i, e in zip(val["caption_id"], val["caption_en"])])
print(f"Baris val dengan caption yang juga ada di train: {int(bocor.sum())}")
if bocor.sum():
    t2, _ = evaluasi(emb, mask=~bocor)
    t2.to_csv(os.path.join(OUT_DIR, "tabel_recall_val_tanpa_overlap.csv"), float_format="%.4f")
    display(t2.round(4))

## 7. Language gap (Recall@1, Inggris dikurangi Indonesia)

In [ ]:
gap = {"gambar->teks": tabel.loc[ARAH[1], "Recall@1"] - tabel.loc[ARAH[0], "Recall@1"],
       "teks->gambar": tabel.loc[ARAH[3], "Recall@1"] - tabel.loc[ARAH[2], "Recall@1"]}
json.dump(gap, open(os.path.join(OUT_DIR, "language_gap.json"), "w"), indent=2)
{k: round(v, 4) for k, v in gap.items()}

## 8. Sepuluh contoh query Indonesia (minimal dua gagal)

In [ ]:
ex = contoh_query(val, emb, ranks[ARAH[2]])
ex.to_csv(os.path.join(OUT_DIR, "contoh_query_id.csv"), index=False)
ex